In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.makedirs('/kaggle/temp/hf', exist_ok=True)

import glob, re, json, time, gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
asr_dirs = glob.glob('/kaggle/input/**/asr_raw', recursive=True)
assert len(asr_dirs) == 1, asr_dirs
ASR_DIR = asr_dirs[0]
WORK = '/kaggle/working'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
assert len(set(uids)) == len(uids) == 985

texts = {}
for u in uids:
    with open(f'{ASR_DIR}/{u.replace("/", "__")}.json') as f:
        r = json.load(f)
    texts[u] = re.sub(r'\s+', ' ', ' '.join(s['text'].strip() for s in r['segments'])).strip()
print('transcripts loaded:', len(texts), '| empty:', sum(1 for t in texts.values() if not t))
assert torch.cuda.is_available(), 'GPU is off'

tok = AutoTokenizer.from_pretrained(MODEL_ID)
assert tok.is_fast, 'need a fast tokenizer for offset mapping'
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model.eval()
print('loaded | layers:', model.config.num_hidden_layers, '| hidden:', model.config.hidden_size)

In [ ]:
# V2: prompts, function, smoke test
from scipy.stats import spearmanr

SYSTEM = 'You are an expert examiner of spoken English grammar.'
MARK = 'Transcript:\n"""\n'
TAIL = 'Answer with a single digit from 1 to 5 and nothing else.'
INTRO = ('Below is an automatic speech-recognition transcript of a 45 to 60 second spoken response. '
         'The transcript comes from speech recognition, so ignore capitalization and punctuation; '
         'repetitions and false starts are real features of the speech.\n\n')

RUBRIC = """1: The speech struggles with proper sentence structure and syntax, with limited control over simple grammatical structures and memorized sentence patterns.
2: Limited understanding of sentence structure and syntax. Simple structures are used, but basic sentence-structure and grammatical mistakes are consistent. Sentences may be left incomplete.
3: Decent grasp of sentence structure but errors in grammatical structure, or decent grasp of grammatical structure but errors in sentence syntax and structure.
4: Strong understanding of sentence structure and syntax, with consistently good control of grammar. Occasional errors are generally minor, do not cause misunderstanding, and most can be self-corrected.
5: High grammatical accuracy and adept control of complex grammar. Grammar is used accurately and effectively, with seldom any noticeable mistakes. Complex structures are handled well and the speaker corrects themselves when necessary."""

TASKS = {
    'rubric': ("Rate ONLY the speaker's grammar (sentence structure, syntax, grammatical accuracy and control of complex structures). "
               'Do not rate vocabulary, pronunciation, fluency or the quality of the ideas.\n\nGrammar score rubric:\n' + RUBRIC),
    'errors': ('Judge ONLY how frequent grammatical errors are (verb forms, agreement, tense, articles, prepositions, word order, broken sentences). '
               'Do not judge vocabulary, pronunciation, fluency or ideas.\n\nScale (higher means MORE errors):\n'
               '1: almost no grammatical errors\n2: occasional minor errors\n3: noticeable errors in some sentences\n'
               '4: frequent errors in many sentences\n5: errors in almost every sentence, or sentences are mostly broken'),
    'wellformed': ('Judge ONLY how well-formed and complete the sentences are. Do not judge vocabulary, pronunciation or ideas.\n\nScale:\n'
                   '1: mostly fragments or broken sentences\n2: often incomplete or malformed sentences\n3: a mix of complete and incomplete sentences\n'
                   '4: mostly complete and well-formed sentences\n5: consistently complete, well-formed sentences'),
    'complexity': ('Judge ONLY how accurately the speaker uses complex grammar (subordinate clauses, relative clauses, conditionals, varied tenses, passives). '
                   'Do not judge vocabulary, pronunciation or ideas.\n\nScale:\n'
                   '1: only memorized or very simple patterns\n2: simple structures with consistent errors\n3: some attempts at complex grammar, with errors\n'
                   '4: good control of complex grammar with minor errors\n5: accurate and effective use of varied complex grammar'),
}

def build(view, text):
    user = INTRO + TASKS[view] + '\n\n' + MARK + text + '\n"""\n\n' + TAIL
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

digit_ids = []
for d in '12345':
    ids = tok.encode(d, add_special_tokens=False)
    assert len(ids) == 1, (d, ids)
    digit_ids.append(ids[0])

POOL_LAYERS = [6, 9, 12, 16, 20, 24]

@torch.inference_mode()
def run_prompt(view, text, pool=False):
    prompt = build(view, text)
    enc = tok(prompt, return_tensors='pt', return_offsets_mapping=pool)
    offs = enc.pop('offset_mapping')[0] if pool else None
    enc = enc.to(model.device)
    out = model(**enc, output_hidden_states=pool, use_cache=False)
    logits = out.logits[0, -1].float()
    lp = torch.log_softmax(logits, -1)
    five = logits[digit_ids].cpu().numpy()
    p = np.exp(five - five.max())
    p /= p.sum()
    res = dict(e=float((p * np.arange(1, 6)).sum()), ent=float(-(p * np.log(p + 1e-12)).sum()),
               mass=float(lp[digit_ids].exp().sum().item()), p=p.tolist())
    if pool:
        a = prompt.rfind(MARK) + len(MARK)
        b = a + len(text)
        assert prompt[a:b] == text, 'transcript span not found'
        m = (offs[:, 0] >= a) & (offs[:, 1] <= b) & (offs[:, 1] > offs[:, 0])
        idx = torch.nonzero(m).squeeze(-1).to(model.device)
        res['ntr'] = int(idx.numel())
        res['pool'] = torch.stack([out.hidden_states[L][0].index_select(0, idx).float().mean(0)
                                   for L in POOL_LAYERS], 0).cpu().numpy()
    return res

def run_all(uid):
    r = {'rubric': run_prompt('rubric', texts[uid], pool=True)}
    for v in ('errors', 'wellformed', 'complexity'):
        r[v] = run_prompt(v, texts[uid])
    return r

lab = train.set_index('filename').label
samp = [u for u in uids[:len(train)] if lab[u.split('/')[1]] > 0][:40]
t0 = time.time()
res = [run_all(u) for u in samp]
sec = (time.time() - t0) / len(samp)
yy = np.array([lab[u.split('/')[1]] for u in samp])
print('seconds per file (4 passes):', round(sec, 2), '| estimated total minutes:', round(sec * 985 / 60, 1))
print('transcript tokens: min', min(r['rubric']['ntr'] for r in res), 'max', max(r['rubric']['ntr'] for r in res),
      '(roughly 100 to 300 expected; a tiny number means the span lookup failed)')
print('pooled shape:', res[0]['rubric']['pool'].shape, '(expect (6, 3584))')
print('NaN present:', not all(np.isfinite(r['rubric']['pool']).all() and all(np.isfinite(r[v]['e']) for v in r) for r in res))
print('largest abs pooled value:', round(float(max(np.abs(r['rubric']['pool']).max() for r in res)), 1))
for v in ('rubric', 'errors', 'wellformed', 'complexity'):
    e = np.array([r[v]['e'] for r in res])
    print(f'{v:11s} digit mass min {min(r[v]["mass"] for r in res):.3f} | expected score range {e.min():.2f} to {e.max():.2f} '
          f'| Spearman with label (n=40, very noisy) {spearmanr(e, yy)[0]:+.2f}')
print('note: the errors view is reverse-coded (higher = more errors), so a negative Spearman is expected there')

In [ ]:
N, D = len(uids), model.config.hidden_size
P_POOL, P_REC = f'{WORK}/_part_pool.npy', f'{WORK}/_part_rec.json'
VIEWS = ('rubric', 'errors', 'wellformed', 'complexity')

if os.path.exists(P_REC):
    pool_all, recs = np.load(P_POOL), json.load(open(P_REC))
    start = len(recs)
    print('resuming from', start)
else:
    pool_all, recs, start = np.zeros((N, len(POOL_LAYERS), D), dtype=np.float16), [], 0

n_clipped = 0
t0 = time.time()
for i in range(start, N):
    r = run_all(uids[i])
    pl = r['rubric']['pool']
    n_clipped += int((np.abs(pl) > 6e4).sum())
    pool_all[i] = np.clip(pl, -6e4, 6e4).astype(np.float16)
    row = {'uid': uids[i], 'ntr': r['rubric']['ntr']}
    for v in VIEWS:
        row[f'{v}_e'], row[f'{v}_ent'], row[f'{v}_mass'] = r[v]['e'], r[v]['ent'], r[v]['mass']
        for k in range(5):
            row[f'{v}_p{k + 1}'] = r[v]['p'][k]
    recs.append(row)
    if (i + 1) % 100 == 0 or i + 1 == N:
        np.save(P_POOL, pool_all)
        json.dump(recs, open(P_REC, 'w'))
        el = time.time() - t0
        print(f'{i + 1}/{N} | {el / 60:.1f} min | est remaining {(N - i - 1) * el / (i + 1 - start) / 60:.1f} min')

print('pooled values clipped at +-60000:', n_clipped)
print('NaN or inf in pooled states:', not np.isfinite(pool_all).all())
S = pd.DataFrame(recs)
print('rows:', len(S), '| NaN in scores:', int(S.drop(columns='uid').isna().sum().sum()),
      '| transcript tokens min/max:', int(S.ntr.min()), int(S.ntr.max()))
S.to_csv(f'{WORK}/llm_views.csv', index=False)
np.save(f'{WORK}/llm_rubric_meanpool.npy', pool_all)
pd.Series(uids).to_csv(f'{WORK}/llm_views_uids.csv', index=False, header=['uid'])
for p in (P_POOL, P_REC):
    os.remove(p)
print('saved', pool_all.shape, '|', sorted(os.listdir(WORK)))

In [ ]:
# V4: sanity aur signal check
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

S = pd.read_csv(f'{WORK}/llm_views.csv')
prev = pd.read_csv(glob.glob('/kaggle/input/**/llm_scores.csv', recursive=True)[0]).set_index('uid')
print('integrity: corr of rubric_e here vs notebook 9 e:', round(float(np.corrcoef(S.rubric_e, prev.loc[S.uid, 'e'])[0, 1]), 5),
      '(expect about 1.0)')

S['split'] = S.uid.str.split('/').str[0]
S['label'] = [lab[u.split('/')[1]] if u.startswith('train/') else np.nan for u in S.uid]
asr_dur = {}
for u in uids:
    with open(f'{ASR_DIR}/{u.replace("/", "__")}.json') as f:
        asr_dur[u] = json.load(f)['duration']
S['dg'] = np.where(S.uid.map(asr_dur) < 40, 'short', np.where(S.uid.map(asr_dur) < 52, 'dur_45s', 'dur_60s'))
trn = S[(S.split == 'train') & (S.label > 0)]

print('\nSpearman(view expected score, label), train non-zero:')
rows = []
for v in VIEWS:
    row = {'view': v, 'all': spearmanr(trn[f'{v}_e'], trn.label)[0]}
    for g in ['dur_45s', 'dur_60s']:
        d = trn[trn.dg == g]
        row[g] = spearmanr(d[f'{v}_e'], d.label)[0]
    rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\ncorrelation between the four expected scores (train non-zero):')
print(trn[[f'{v}_e' for v in VIEWS]].corr().round(2).to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, v in zip(ax, ['errors', 'wellformed', 'complexity']):
    sns.scatterplot(data=trn, x=f'{v}_e', y='label', hue='dg', alpha=0.4, ax=a)
    a.set_title(f'{v} view vs label')
plt.tight_layout(); plt.show()